In [80]:
import csv

import orthodb

In [81]:
api = orthodb.OdbAPI()

In [83]:
help(orthodb.OdbAPI)

Help on class OdbAPI in module orthodb.odbapi:

class OdbAPI(orthodb.client.Client)
 |  OdbAPI(uri_version='current')
 |
 |  OrthoDB API client.
 |
 |  Params
 |  ------
 |  uri_version     : version string for constructing the URL (default 'current')
 |
 |  As of this version the allowed version strings are:
 |     current, v8, v9, v9.1, v10.1, v11, v12
 |
 |  Those without subversion can also be given using '.0'.
 |  E.g 'v11' and 'v11.0' are equivalent.
 |
 |  Note that at the moment the backward compatability is not enforced. However in most cases
 |  the query response can be obtained by using the `raw_data()` member of the object returned.
 |
 |  Overview
 |  --------
 |  query              raw query interface with OrthoDB API
 |  version            returns the orthodb release id from the API (cached)
 |  tree               get the tree - returns a Tree object with additional functionalities
 |  species            get a list of species (organism ids)
 |  search             make a

In [98]:
q = api.query("uvsx")

ERROR:root:API command uvsx not valid


In [149]:
from collections import Counter
import json

# Load JSON
with open("../data/uniprot/viral_recA_uvsX_all_metadata.json") as f:
    data = json.load(f)

# Collect all GO IDs across all proteins
all_go_ids = []
for entry in data:
    for xref in entry.get("uniProtKBCrossReferences", []):
        if xref.get("database") == "GO":
            go_id = xref.get("id")
            if go_id:
                all_go_ids.append(go_id)

# Count how many proteins each GO ID occurs in
go_counts = Counter(all_go_ids)

# Show results
for go_id, count in go_counts.most_common(10):
    print(f"{go_id}: {count} occurrences")

GO:0003677: 4163 occurrences
GO:0006310: 4065 occurrences
GO:0015074: 3147 occurrences
GO:0044826: 2443 occurrences
GO:0075713: 2441 occurrences
GO:0016740: 2391 occurrences
GO:0016787: 1992 occurrences
GO:0005524: 1057 occurrences
GO:0006281: 1035 occurrences
GO:0003697: 993 occurrences


In [152]:
import csv
import requests

# Path to your metadata CSV
csv_path = "../data/ncbi/uvsx_metadata.csv"

# Read UniProt accessions from CSV
uni_proteins = []
with open(csv_path, newline='') as f:
    reader = csv.DictReader(f)
    for row in reader:
        accession = row.get("accession")
        if accession:
            uni_proteins.append(accession)

print(f"Found {len(uni_proteins)} accessions in CSV")

# Query OrthoDB for each accession to get OG IDs
accession_to_og = {}

for accession in uni_proteins:
    url = f"https://www.orthodb.org/protein/{accession}?format=json"
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        og_id = data.get("ogid")
        if og_id:
            accession_to_og[accession] = og_id
        else:
            print(f"{accession} has no OG ID in OrthoDB")
    else:
        print(f"{accession} not found in OrthoDB (status {response.status_code})")

print(f"Mapped {len(accession_to_og)} accessions to OG IDs")

# Example: print first 10 mappings
for acc, og in list(accession_to_og.items())[:10]:
    print(f"{acc} → {og}")

Found 17858 accessions in CSV
P32270 not found in OrthoDB (status 404)
P20315 not found in OrthoDB (status 404)
UYE93698 not found in OrthoDB (status 404)
YDI96965 not found in OrthoDB (status 404)
YDI96730 not found in OrthoDB (status 404)
YDI94796 not found in OrthoDB (status 404)
YDI94551 not found in OrthoDB (status 404)
YDI93881 not found in OrthoDB (status 404)
YDI88709 not found in OrthoDB (status 404)
YCJ21993 not found in OrthoDB (status 404)
YDF88887 not found in OrthoDB (status 404)
YDF81549 not found in OrthoDB (status 404)
YDF76131 not found in OrthoDB (status 404)
YDF75915 not found in OrthoDB (status 404)
YDF71083 not found in OrthoDB (status 404)
YDF68759 not found in OrthoDB (status 404)
YDF68054 not found in OrthoDB (status 404)
YDF61728 not found in OrthoDB (status 404)
YDD38469 not found in OrthoDB (status 404)
YDD21067 not found in OrthoDB (status 404)
BHD21371 not found in OrthoDB (status 404)
YCZ41734 not found in OrthoDB (status 404)
QVG64171 not found in OrthoD

ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))